In [1]:
!pip install -q transformers datasets accelerate scikit-learn torch

In [2]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_csv("/content/tweet_emotions_ptbr_treated.csv")

df['BASE_TEXT'] = (
    df['BASE_TEXT']
    .fillna("")        # remove NaN
    .astype(str)       # garante string
)

le = LabelEncoder()
df["label"] = le.fit_transform(df["EMOTION"])


num_labels = len(le.classes_)

print(df.shape)
print("Number of emotions:", num_labels)
df.head(3)

(12419, 8)
Number of emotions: 16


,texto,EMOTION,UNCLEAN_TEXT,BASE_TEXT,TEXT_NO_STOP,TEXT_LEMMA,CLEAN_TEXT,label
0,͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏\nque júbilo incalculável ver t...,alegria,͏ ͏ ͏ ͏ ͏ ͏ ͏ ͏\nque júbilo incalculável ver t...,que jubilo incalculavel ver todos meus querido...,jubilo incalculavel queridos prestaram ano pas...,que jubilo incalculavel ver todo meu querido q...,jubilo incalculavel querido prestar ano passad...,1
1,". 𝘀𝗮𝗽𝗼𝗻𝘁𝗮𝗺𝗲𝗻𝘁𝗼𝗶\nSenhor, arranca do meu coraçã...",decepção,". 𝘀𝗮𝗽𝗼𝗻𝘁𝗮𝗺𝗲𝗻𝘁𝗼𝗶\nSenhor, arranca do meu coraçã...",sapontamentoi senhor arranca meu coracao qualq...,sapontamentoi senhor arranca coracao desaponta...,sapontamentoir senhor arrancar meu coracao qua...,sapontamentoir senhor arrancar coracao desapon...,5
2,"A franquia continua ativa, com o desenvolvimen...",decepção,"A franquia continua ativa, com o desenvolvimen...",franquia continua ativa com desenvolvimento jo...,franquia continua ativa desenvolvimento jogo p...,franquia continuar ativo com desenvolvimento j...,franquia continuar ativo desenvolvimento jogo ...,5


In [4]:
X = df["BASE_TEXT"]
y = df["label"]

X_train, X_aux, y_train, y_aux = train_test_split(
    X, y,
    train_size=0.70,
    random_state=42,
    stratify=y
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_aux, y_aux,
    train_size=0.50,
    random_state=42,
    stratify=y_aux
)

df_train = pd.DataFrame({"text": X_train, "label": y_train})
df_dev   = pd.DataFrame({"text": X_dev,   "label": y_dev})
df_test  = pd.DataFrame({"text": X_test,  "label": y_test})

train_ds = Dataset.from_pandas(df_train)
dev_ds   = Dataset.from_pandas(df_dev)
test_ds  = Dataset.from_pandas(df_test)

In [5]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
dev_ds   = dev_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format("torch")
dev_ds.set_format("torch")
test_ds.set_format("torch")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Map:   0%|          | 0/8693 [00:00<?, ? examples/s]

Map:   0%|          | 0/1863 [00:00<?, ? examples/s]

Map:   0%|          | 0/1863 [00:00<?, ? examples/s]

In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )

    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels, preds, average="micro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
    }

In [7]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="/content/bertimbau_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
    fp16=True,
    lr_scheduler_type="linear",
    warmup_ratio=0.2,
    seed=42,
    data_seed=42
)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

In [8]:
trainer.train()

test_results = trainer.evaluate(test_ds)
test_results

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted,Precision Micro,Recall Micro,F1 Micro
1,0.363506,0.177151,0.962426,0.964807,0.962608,0.963172,0.964271,0.962426,0.962805,0.962426,0.962426,0.962426
2,0.138817,0.118903,0.966184,0.967496,0.966262,0.966539,0.967667,0.966184,0.966582,0.966184,0.966184,0.966184
3,0.068653,0.128628,0.964037,0.964906,0.964284,0.964309,0.964463,0.964037,0.963965,0.964037,0.964037,0.964037
4,0.046239,0.137446,0.967257,0.967973,0.967409,0.967500,0.967594,0.967257,0.967236,0.967257,0.967257,0.967257


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.19099493324756622,
 'eval_accuracy': 0.9570585077831455,
 'eval_precision_macro': 0.9577165890211798,
 'eval_recall_macro': 0.9576644510742872,
 'eval_f1_macro': 0.9575074327678856,
 'eval_precision_weighted': 0.9572851908213489,
 'eval_recall_weighted': 0.9570585077831455,
 'eval_f1_weighted': 0.9569861808829084,
 'eval_precision_micro': 0.9570585077831455,
 'eval_recall_micro': 0.9570585077831455,
 'eval_f1_micro': 0.9570585077831455,
 'eval_runtime': 2.2053,
 'eval_samples_per_second': 844.791,
 'eval_steps_per_second': 53.055,
 'epoch': 4.0}

In [9]:
pred_output = trainer.predict(test_ds)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print("\n===== MÉTRICAS POR EMOÇÃO =====")

report = classification_report(
    y_true,
    y_pred,
    target_names=le.classes_,
    digits=2,
    zero_division=0
)

print(report)


===== MÉTRICAS POR EMOÇÃO =====
               precision    recall  f1-score   support

agressividade       0.89      0.92      0.90       119
      alegria       0.93      0.91      0.92       120
         amor       0.91      0.97      0.94       117
  antecipacao       0.99      1.00      1.00       109
    confianca       0.93      0.97      0.94       115
     decepção       0.96      0.94      0.95       117
     desprezo       1.00      0.97      0.99       120
  intimidação       1.00      1.00      1.00       105
         medo       0.99      0.98      0.99       118
         nojo       0.96      0.97      0.97       118
     otimismo       0.98      0.99      0.99       121
        raiva       0.98      0.97      0.98       119
      remorso       0.98      0.99      0.99       116
    submissão       0.97      0.95      0.96       119
     surpresa       0.95      0.94      0.94       109
     tristeza       0.90      0.85      0.88       121

     accuracy                 

## (2) BERT Com CB Loss

In [10]:
X = df["BASE_TEXT"]
y = df["label"]

X_train, X_aux, y_train, y_aux = train_test_split(
    X, y,
    train_size=0.70,
    random_state=42,
    stratify=y
)

X_dev, X_test, y_dev, y_test = train_test_split(
    X_aux, y_aux,
    train_size=0.50,
    random_state=42,
    stratify=y_aux
)

df_train = pd.DataFrame({"text": X_train, "label": y_train})
df_dev   = pd.DataFrame({"text": X_dev,   "label": y_dev})
df_test  = pd.DataFrame({"text": X_test,  "label": y_test})

train_ds = Dataset.from_pandas(df_train)
dev_ds   = Dataset.from_pandas(df_dev)
test_ds  = Dataset.from_pandas(df_test)

In [11]:
num_classes = len(np.unique(y_train))

samples_per_class = np.bincount(y_train, minlength=num_classes)

beta = 0.999
effective_num = 1.0 - np.power(beta, samples_per_class)
weights = (1.0 - beta) / effective_num
weights = weights / np.sum(weights) * num_classes

In [12]:
model_name = "neuralmind/bert-base-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize, batched=True)
dev_ds   = dev_ds.map(tokenize, batched=True)
test_ds  = test_ds.map(tokenize, batched=True)

train_ds.set_format("torch")
dev_ds.set_format("torch")
test_ds.set_format("torch")

Map:   0%|          | 0/8693 [00:00<?, ? examples/s]

Map:   0%|          | 0/1863 [00:00<?, ? examples/s]

Map:   0%|          | 0/1863 [00:00<?, ? examples/s]

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro"
    )

    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds, average="weighted"
    )

    precision_micro, recall_micro, f1_micro, _ = precision_recall_fscore_support(
        labels, preds, average="micro"
    )

    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
        "precision_micro": precision_micro,
        "recall_micro": recall_micro,
        "f1_micro": f1_micro,
    }

In [14]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

training_args = TrainingArguments(
    output_dir="/content/bertimbau_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=4,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    logging_steps=100,
    report_to="none",
    fp16=True,
    lr_scheduler_type="linear",
    warmup_ratio=0.2,
    seed=42,
    data_seed=42
)

data_collator = DataCollatorWithPadding(tokenizer)

class CBTrainer(Trainer):
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights).float()

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=0):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        device = logits.device
        labels = labels.to(device)

        weights = self.class_weights.to(device)

        loss_fct = torch.nn.CrossEntropyLoss(weight=weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

trainer = CBTrainer(
    class_weights=weights,
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. C

In [15]:
trainer.train()

test_results = trainer.evaluate(test_ds)
test_results

Epoch,Training Loss,Validation Loss,Accuracy,Precision Macro,Recall Macro,F1 Macro,Precision Weighted,Recall Weighted,F1 Weighted,Precision Micro,Recall Micro,F1 Micro
1,0.399862,0.182759,0.960279,0.964930,0.960370,0.961537,0.964330,0.960279,0.961173,0.960279,0.960279,0.960279
2,0.137148,0.115948,0.964573,0.966086,0.964872,0.965012,0.966166,0.964573,0.964901,0.964573,0.964573,0.964573
3,0.069369,0.120797,0.968867,0.969613,0.968982,0.969115,0.969361,0.968867,0.968931,0.968867,0.968867,0.968867
4,0.043351,0.129833,0.967257,0.967723,0.967392,0.967421,0.967456,0.967257,0.967220,0.967257,0.967257,0.967257


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

{'eval_loss': 0.16590803861618042,
 'eval_accuracy': 0.9634997316156736,
 'eval_precision_macro': 0.9644181776708314,
 'eval_recall_macro': 0.9639455259370973,
 'eval_f1_macro': 0.9639851287659835,
 'eval_precision_weighted': 0.9640725649611093,
 'eval_recall_weighted': 0.9634997316156736,
 'eval_f1_weighted': 0.96358571701315,
 'eval_precision_micro': 0.9634997316156736,
 'eval_recall_micro': 0.9634997316156736,
 'eval_f1_micro': 0.9634997316156736,
 'eval_runtime': 2.2172,
 'eval_samples_per_second': 840.264,
 'eval_steps_per_second': 52.77,
 'epoch': 4.0}

In [16]:
pred_output = trainer.predict(test_ds)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=1)

print("\n===== MÉTRICAS POR EMOÇÃO =====")

report = classification_report(
    y_true,
    y_pred,
    target_names=le.classes_,
    digits=2,
    zero_division=0
)

print(report)


===== MÉTRICAS POR EMOÇÃO =====
               precision    recall  f1-score   support

agressividade       0.88      0.96      0.92       119
      alegria       0.96      0.93      0.94       120
         amor       0.93      0.97      0.95       117
  antecipacao       1.00      1.00      1.00       109
    confianca       0.93      0.97      0.95       115
     decepção       0.97      0.97      0.97       117
     desprezo       1.00      0.97      0.99       120
  intimidação       1.00      1.00      1.00       105
         medo       0.99      0.99      0.99       118
         nojo       0.97      0.97      0.97       118
     otimismo       0.98      0.99      0.99       121
        raiva       0.98      0.97      0.98       119
      remorso       0.98      0.98      0.98       116
    submissão       0.99      0.95      0.97       119
     surpresa       0.94      0.94      0.94       109
     tristeza       0.90      0.88      0.89       121

     accuracy                 